In [57]:
import re
from util_ZENDB import *
import pandas as pd
import random
from cal_sel import *
#from llm4 import *


from llm import *
# Your API key
#OPENAI_KEY = 'sk-X5lP13FSF9jrOFMx7d3dC460E9B94dB792A25c0cF140EeBa'
OPENAI_KEY = 'sk-1Zssb4A7L2lFfna7D109859e3f014c23B9Ed567e602dEf46'

# Initialize the OpenAI API
init_chatgpt(OPENAI_KEY)

#data是从./university/ground_truth.csv中读取的
data = pd.read_csv("./datalake/ground_truth/university.csv")


#data是其中sample的10个数据
data = data.sample(99)
#针对data中的每一个值进行判断是不是数字，是的话转为int
for column in data.columns:
    data[column] = data[column].apply(lambda x: int(x) if isinstance(x, str) and isDigit(x) else x)


In [58]:
sql_query = """
SELECT university_name, academic_calendar, rank FROM university_data WHERE academic_calendar = 4-1-4-based AND (year_founded < 1900 OR total_undergraduate_enrollment < 1500) AND rank < 20
"""
unidir = './result/university/main/Q3/sql3/'
if not os.path.exists(unidir):
    os.makedirs(unidir)
where_clause = parse_sql(sql_query)
where_filter_condition = extract_filter_condition(where_clause)
print("where_filter_condition",where_filter_condition)
where_filter_attribute = []
for condition in where_filter_condition:
    attr = condition.split()[0]
    if attr not in where_filter_attribute:
        where_filter_attribute.append(attr)
print("where_filter_attribute",where_filter_attribute)

select_clause = re.search(r'SELECT (.+?) FROM', sql_query, re.IGNORECASE).group(1)
select_attributes = [attr.strip() for attr in select_clause.split(',')]
print("select_attributes",select_attributes)
####output是dataframe，其中的属性是select_attributes
output = pd.DataFrame(columns=select_attributes)

where_filter_condition ['academic_calendar = 4-1-4-based', 'year_founded < 1900', 'total_undergraduate_enrollment < 1500', 'rank < 20']
where_filter_attribute ['academic_calendar', 'year_founded', 'total_undergraduate_enrollment', 'rank']
select_attributes ['university_name', 'academic_calendar', 'rank']


In [59]:
def parse_allinput(input_text, attributes):
    attribute_blocks = input_text.strip().split('\n\n')
    for block in attribute_blocks:
        lines = block.strip().split('\n')
        name = lines[0].strip()
        key_sentences = []
        for line in lines[1:]:
            if ':' in line:
                key_sentences.append(line.split(':')[1].strip())
            else:
                key_sentences.append(line)
        # 如果 attributes 中没有 name，那么就添加 name 并创建字典结构
        if name not in attributes:
            attributes[name] = {
                'key_sentences': key_sentences
            }
        else:
            attributes[name]['key_sentences'].extend(key_sentences)
    
    return attributes

def calculate_s_all(selectivity, key_sentences, n , formula='default'):
    total_tokens = sum(len(re.findall(r'\w+', sentence)) for sentence in key_sentences)
    if formula == 'default':
        s_value = (1 - selectivity) / (total_tokens/n)
    elif formula == 'alternate':
        s_value = selectivity / (total_tokens/n)
    return s_value

In [60]:
filter_cond = extract_filter_condition(where_clause)

file_list = os.listdir("./datalake/university")
attributes_all = {}
filter_data_all = {}
file_num = 0
for onefile in file_list:
    file_num += 1
    input_text = ""
    file_path = "./datalake/university/" + onefile
    with open(file_path, 'r') as file:
        input_text = file.read()
    attributes_all = parse_allinput(input_text, attributes_all)
    

for filter_condtion in filter_cond:
        atrribute,operator,value = filter_condtion.split(maxsplit=2)
        #print(f"atrribute: {atrribute}, operator: {operator}, value: {value}")
        selectivity = cal_sel(data, {"name": filter_condtion})
        if atrribute in attributes_all:
            filter_data_all[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': calculate_s_all(selectivity, attributes_all[atrribute]['key_sentences'], n = file_num),
                "key_sentences": attributes_all[atrribute]['key_sentences']
            }
        else:
            filter_data_all[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': 0,
                "key_sentences": []
            }
    #print("filter_data: ",filter_data)

sorted_filters = handle_sql(where_clause, filter_data_all)
print("sorted_filters",sorted_filters)

where_clause:  academic_calendar = 4-1-4-based AND (year_founded < 1900 OR total_undergraduate_enrollment < 1500) AND rank < 20
composite_expressions:  ['(year_founded < 1900 OR total_undergraduate_enrollment < 1500)']

Processing composite_expression:  (year_founded < 1900 OR total_undergraduate_enrollment < 1500)
internal_filters for composite_expression 1: ['year_founded < 1900', 'total_undergraduate_enrollment < 1500']
S value for composite_expression 1: 26.0496

Remaining Expression Parts: [['academic_calendar = 4-1-4-based'], 'AND', ['composite_filter_1'], 'AND', ['rank < 20']]
Best composite filters: [['academic_calendar = 4-1-4-based', 'composite_filter_1'], 0.0, 'rank < 20']
Remaining filters: ['academic_calendar = 4-1-4-based', 'composite_filter_1', 'rank < 20']
expression: (year_founded < 1900 OR total_undergraduate_enrollment < 1500), s_value: 26.049586776859506
Internal sorted filters for (year_founded < 1900 OR total_undergraduate_enrollment < 1500): [('total_undergraduat

In [61]:
all_total_token = 0
all_actual_token = 0
import os
import time
infor = unidir + "infor.txt"
file_list = os.listdir("./datalake/university")
start_tie = time.time()

for onefile in file_list:
    ###############################################################
    ########################这里是sql更新的部分#######################
    ###############################################################
    
    
    input_text = ""
    file_path = "./datalake/university/" + onefile
    print("file: ",file_path)
    with open(file_path, "r") as file:
        input_text = file.read()

    #提取filter_conditions
    
    #print("where_clause: ",where_clause)

    ###########################################
    ########这里是手动计算selectivity的部分########
    ###########################################
    
    #print("filter_cond: ",filter_cond)

    #print("text: ",input_text)
    attributes = parse_input(input_text)

    #计算total token
    total_token = 0
    total_token += len(input_text.split()) + 100
    
    print("total_token: ",total_token)
    all_total_token += total_token
    print("all_total_token: ",all_total_token)

    print("\n")
    
    actual_token = 0
    ####询问大模型阶段
    print("###################询问大模型阶段#########################\n")

    skipthefile = 0
    sql_copy = sql_query
    #remian_attributes是全局的，存储剩余的属性
    remaining_attributes = select_attributes.copy()
    curdata = {}
    sorted_filter_cond = []
    mapfilter_cond = {}

    for filter in sorted_filters:
        sorted_filter_cond.append(filter[0])
    for filter in sorted_filter_cond:
        mapfilter_cond[filter] = 0
    #给一个容器存储剩余的filter条件
    remianing_sorted_filter_cond = sorted_filter_cond.copy()
    # for a in sorted_filter_cond:
    #     print(a)
    while remianing_sorted_filter_cond:
        #print("remianing_sorted_filter_cond: ",remianing_sorted_filter_cond)
        filter_cond_tochange = []
        filter_cond = remianing_sorted_filter_cond[0]
        print("filter_cond: ",filter_cond)
        mapfilter_cond[filter_cond] += 1
        if mapfilter_cond[filter_cond] >= 2:
            if filter_cond in remianing_sorted_filter_cond:
                remianing_sorted_filter_cond.remove(filter_cond)
                tochange_filter = [filter_cond,'false']
                filter_cond_tochange.append(tochange_filter)
            continue

        elif mapfilter_cond[filter_cond] == 1:
            attribute = filter_cond.split()[0].strip()
            key_sentences = ""
            for sentence in attributes[attribute]['key_sentences']:
                key_sentences += sentence
            print("key_sentence: ",key_sentences)
            
            answer = ask_completion4filtercondANDattr(str(remaining_attributes),str(remianing_sorted_filter_cond),key_sentences)
            actual_token += len(key_sentences.split())+100
            answer = remove_punctuation(answer)
            #print("answer: \n", answer)
            try:
                attributes_cur_all = answer.split("$$")[1]
                filter_cond_cur_all = answer.split("$$")[0]
                print("attributes_cur_all: ", attributes_cur_all)
                print("filter_cond_cur_all: ", filter_cond_cur_all)
            except:
                print("error")
                print("answer: ", answer)
                continue

            #处理属性的抽取
            try:
                minidatas = attributes_cur_all.split("##")
                for minidata in minidatas:
                    dataattr,datavalue = minidata.split(":")
                    curdata[dataattr] = datavalue
                    if dataattr in remaining_attributes:
                        if 'NAN' not in datavalue:
                            remaining_attributes.remove(dataattr)
                            print("remaining_attributes: ",remaining_attributes)
            except:
                print("###############")
                print("error attributes")
                print("attribute_cur_all: ", attributes_cur_all)
            #处理filter_cond的抽取
            try:
                minifilters = filter_cond_cur_all.split("##")
                for minifilter in minifilters:
                    filter_cond_cur,bool_value = minifilter.split(":")
                    tochange_filter = [filter_cond_cur,bool_value]
                    if 'NAN' not in bool_value:
                        filter_cond_tochange.append(tochange_filter)
                        print("filter_cond: ",filter_cond_cur)
                        if filter_cond_cur in remianing_sorted_filter_cond:
                            remianing_sorted_filter_cond.remove(filter_cond_cur)
                        print("remianing_sorted_filter_cond: ",remianing_sorted_filter_cond)
            except:
                print("###############")
                print("error filter_cond")
                actual_token -= len(key_sentences.split())+100
                print("filter_cond_cur_all: ", filter_cond_cur_all)
        
        #将answer中的值填入到sql_query中
        for ask_filter_cond,bool_value in filter_cond_tochange:
            sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
        sql_copy2 = sql_copy
        print("sql_copy: ",sql_copy)
        bool_value_set_true = calculate_bool_value_true(sql_copy)
        bool_value_set_false = calculate_bool_value_false(sql_copy2)
        print("bool_value_set_true: ",bool_value_set_true)
        print("bool_value_set_false: ",bool_value_set_false)
        #全设true为false就跳出该文本
        if bool_value_set_true == False:
            skipthefile = 1
            break
        #全设false为true就跳出该文本，因为已经找到了一个解
        if bool_value_set_false == True:
            skipthefile = 0
            break
        else:
            skipthefile = 1
        print("\n")
        #remianing_sorted_filter_cond.remove(filter_cond)



    #如果skipthefile为0，则从`SELECT`子句中抽取对应属性的key sentences，使用大模型得到值并存储
    if skipthefile == 0:
        print("###################抽取对应的属性#########################\n")
        
        key_sentences = ""
        #做一个attribute的map，如果第二次以他为key sentence的时候，就不再问了
        mapattr = {}
        for attr in select_attributes:
            mapattr[attr] = 0
        print("remaining_attributes: ",remaining_attributes)
        while remaining_attributes:
            #attr = random.choice(remaining_attributes)
            #选择第一个属性
            attr = remaining_attributes[0]
            mapattr[attr] += 1
            if mapattr[attr] == 2:
                remaining_attributes.remove(attr)
                curdata[attr] = "NAN"
                continue
            if attr in attributes:
                key_sentences = ""
                for sentence in attributes[attr]['key_sentences']:
                    key_sentences += sentence + " "
                print("attribute: ",attr)
                print("key_sentences: ", key_sentences)
                answer = ask_completion4Multattribute(str(remaining_attributes), key_sentences)
                actual_token += len(key_sentences.split())+100
                answer = remove_punctuation(answer)
                print("answer: \n", answer)
                try:
                    minidatas = answer.split("##")
                    for minidata in minidatas:
                        dataattr,datavalue = minidata.split(":")
                        curdata[dataattr] = datavalue
                        if dataattr in remaining_attributes:
                            if 'NAN' not in datavalue:
                                remaining_attributes.remove(dataattr)
                except:
                    print("error")
                    print("answer: ", answer) 
             
        if len(curdata) != 0:
            curdata['file'] = onefile
            output = output.append(curdata,ignore_index=True)
        #删除全为NAN的行
        output.dropna(axis=0, how='all', inplace=True) 
    
    print("actual_token: ",actual_token)
    all_actual_token += actual_token
    print("all_actual_token: ",all_actual_token)
    
 

end_time = time.time()
print("all_actual_token: ",all_actual_token)
print("all_total_token: ",all_total_token)
print("time: ",end_time-start_tie)

with open(infor, "w") as file:
    file.write("all_actual_token: "+str(all_actual_token)+"\n")
    file.write("all_total_token: "+str(all_total_token)+"\n")
    file.write("time: "+str(end_time-start_tie)+"\n")

file:  ./datalake/university/0173.txt
total_token:  1507
all_total_token:  1507


###################询问大模型阶段#########################

filter_cond:  total_undergraduate_enrollment < 1500
key_sentence:  Jamestown College The Basics About Jamestown College Jamestown College is a private institution that was founded in 1884.Jamestown College's ranking in the 2011 edition of Best Colleges is Regional Colleges (Midwest), 41.Use of this Web site constitutes acceptance of our Terms and Conditions of Use and Privacy Policy.Copyright © 2011 U.S. News & World Report, L.P. All rights reserved.It has a total undergraduate enrollment of 1,010, its setting is urban, and the campus size is 110 acres.It utilizes a semester-based academic calendar.Its tuition and fees are $13,870.It utilizes a semester-based academic calendar.
attributes_cur_all:  university_name:Jamestown College##academic_calendar:semester-based##rank:41
filter_cond_cur_all:  total_undergraduate_enrollment < 1500:true##year_founded <

In [62]:
import pandas as pd

# 读取数据
all_data = pd.read_csv("./datalake/ground_truth/university.csv")
#print(all_data.dtypes)

# 使用遍历方式过滤数据
filtered_rows = []
for index, row in all_data.iterrows():
    cur = {}
    university_name = row['university_name']
    year_founded = row['year_founded']
    rank = row['rank']
    setting = row['setting']
    total_undergraduate_enrollment = row['total_undergraduate_enrollment']
    academic_calendar = row['academic_calendar']
    campus_size = row['campus_size']
    file = row['file']

    if not isinstance(university_name, str):
        university_name = str(university_name)

    if not isinstance(setting, str):
        setting = str(setting)

    if not isinstance(academic_calendar, str):
        academic_calendar = str(academic_calendar)

    if not isinstance(file, str):
        file = str(file)
        if len(file) == 1:
            file = '000' + file
        elif len(file) == 2:
            file = '00' + file
        elif len(file) == 3:
            file = '0' + file
    
    if not isinstance(campus_size, str):
        campus_size = str(campus_size)
    
    if pd.isna(total_undergraduate_enrollment) or total_undergraduate_enrollment in ['N/A', ' ', '']:
        total_undergraduate_enrollment = 10000
    else:
        try:
            total_undergraduate_enrollment = int(total_undergraduate_enrollment)
        except ValueError:
            total_undergraduate_enrollment = 10000

    if pd.isna(year_founded) or year_founded in ['N/A', ' ', '']:
        year_founded = 10000
    else:
        try:
            year_founded = int(year_founded)
        except ValueError:
            year_founded = 10000

    if pd.isna(rank) or rank in ['N/A', ' ', '']:
        rank = 10000
    else:
        try:
            rank = int(rank)
        except ValueError:
            rank = 10000
    
    cur['university_name'] = university_name
    cur['year_founded'] = year_founded
    cur['rank'] = rank
    cur['setting'] = setting
    cur['total_undergraduate_enrollment'] = total_undergraduate_enrollment
    cur['academic_calendar'] = academic_calendar
    cur['campus_size'] = campus_size
    cur['file'] = file + ".txt"
    

    if academic_calendar == '4-1-4-based' and (year_founded < 1900 or total_undergraduate_enrollment < 1500) and rank < 20:
        filtered_rows.append(cur)

print(filtered_rows)            

# 将过滤后的数据转换为 DataFrame
filtered_data = pd.DataFrame(filtered_rows)

# 选择需要的列
true_output = filtered_data[['university_name', 'academic_calendar', 'year_founded', 'file']]

#删除重复的行
true_output.drop_duplicates(inplace=True)

true_output

[{'university_name': 'Union University', 'year_founded': 1823, 'rank': 15, 'setting': 'urban', 'total_undergraduate_enrollment': 2780, 'academic_calendar': '4-1-4-based', 'campus_size': '360 acres', 'file': '0630.txt'}, {'university_name': 'Massachusetts Maritime Academy', 'year_founded': 1891, 'rank': 13, 'setting': 'Suburban', 'total_undergraduate_enrollment': 1185, 'academic_calendar': '4-1-4-based', 'campus_size': '55 acres', 'file': '0255.txt'}, {'university_name': 'Williams College', 'year_founded': 1793, 'rank': 1, 'setting': 'rural', 'total_undergraduate_enrollment': 2067, 'academic_calendar': '4-1-4-based', 'campus_size': '450 acres', 'file': '0884.txt'}, {'university_name': 'Converse College', 'year_founded': 1889, 'rank': 15, 'setting': 'urban', 'total_undergraduate_enrollment': 729, 'academic_calendar': '4-1-4-based', 'campus_size': '70 acres', 'file': '0975.txt'}, {'university_name': 'Hamline University', 'year_founded': 1854, 'rank': 9, 'setting': 'urban', 'total_undergra

/var/folders/4p/pkgfgx150g18jgpd38wqxctm0000gn/T/ipykernel_10747/203757741.py:87: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  true_output.drop_duplicates(inplace=True)


,university_name,academic_calendar,year_founded,file
0,Union University,4-1-4-based,1823,0630.txt
1,Massachusetts Maritime Academy,4-1-4-based,1891,0255.txt
2,Williams College,4-1-4-based,1793,0884.txt
3,Converse College,4-1-4-based,1889,0975.txt
4,Hamline University,4-1-4-based,1854,0126.txt
5,Bethel University,4-1-4-based,1871,0319.txt


In [63]:
#删去全为NAN的列
output.dropna(axis=0, how='all', inplace=True)
output

,university_name,academic_calendar,rank,file
0,Williams College,4-1-4-based,1,0884.txt
1,Hamline University,4-1-4-based,9,0126.txt
2,Bethel University,4-1-4-based,17,0319.txt
3,Converse College,4-1-4-based,15,0975.txt
4,Union University,4-1-4-based,15,0630.txt


In [64]:
#保存output到csv文件
output_path = unidir + "output.csv"
true_output_path = unidir + "true_output.csv"
output.to_csv(output_path, index=False)
true_output.to_csv(true_output_path, index=False)

In [65]:
# #计算output和true_output的file列有多少一样的,true_num是true_output中的行数
true_num = pd.read_csv(true_output_path).shape[0]
num_output = pd.read_csv(output_path).shape[0]
#print("true_num: ",true_num)
num_true = 0
#print(true_output['file'].values)
for index, row in output.iterrows():
    file = row['file']
    if file in true_output['file'].values:
        num_true += 1
    else:
        print(file)
print(f"out_true: {num_true}, out_false: {num_output-num_true}, true_num: {true_num}")
precession = num_true/num_output
recall = num_true/true_num
f1 = 2*precession*recall/(precession+recall)
print(f"precession:{precession}, recall:{recall}, f1:{f1}")

acc_path = unidir + "acc.txt"
sql=sql_query
with open(acc_path, "w") as file:
    file.write(sql)
    file.write(f"out_true: {num_true}, out_false: {num_output-num_true}, true_num: {true_num}\n")
    file.write(f"precession:{precession}, recall:{recall}, f1:{f1}\n")

out_true: 5, out_false: 0, true_num: 6
precession:1.0, recall:0.8333333333333334, f1:0.9090909090909091
